# Lab 07.3: Silver → Gold - KPIs de Negocio
## TransCore Data Engineer - Módulo 07

**Objetivo**: Cargar modelo dimensional desde silver/, calcular KPIs agregados y guardar en gold/.

**Input**: Modelo dimensional en `s3://transcore-infra-prod-eu-west-1-ric/silver/obra-lineal/año=2026/mes=05/dia=06/`

**Output**: KPIs mensuales en `s3://transcore-infra-prod-eu-west-1-ric/gold/obra-lineal/`
- `kpis_mensuales_activos_2026-05.parquet`

## 1. Configuración AWS

In [ ]:
# Configuración de credenciales AWS (S3 real)
import os

os.environ["AWS_ACCESS_KEY_ID"] = "<TU_ACCESS_KEY_ID>"
os.environ["AWS_SECRET_ACCESS_KEY"] = "<TU_SECRET_ACCESS_KEY>"

# Limpiar cualquier configuración de LocalStack
os.environ.pop("AWS_ENDPOINT_URL", None)
os.environ.pop("AWS_ENDPOINT", None)

# Configurar región de AWS explícitamente para AWS real
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

print("Configuración: AWS real, región eu-west-1")

## 2. Importar Libraries

In [ ]:
import numpy as np
import pandas as pd
import boto3
from datetime import datetime
from pathlib import Path
import tempfile
from io import BytesIO

print("Librerías importadas correctamente")

## 3. Configuración de S3 y Paths

In [ ]:
# Configuración S3
BUCKET_NAME = "<TU_BUCKET_NAME>"
FECHA_PROCESO = "2026-05-06"
MES_PROCESO = "2026-05"

# Paths
SILVER_PREFIX = f"silver/obra-lineal/año=2026/mes=05/dia=06"
GOLD_PREFIX = "gold/obra-lineal"

# Crear cliente S3
s3_client = boto3.client("s3", region_name="eu-west-1")

print(f"Bucket: {BUCKET_NAME}")
print(f"Silver prefix: {SILVER_PREFIX}")
print(f"Gold prefix: {GOLD_PREFIX}")

## 4. Cargar Modelo Dimensional desde Silver/

In [ ]:
def load_parquet_from_s3(bucket, key):
    """Carga un Parquet directamente desde S3 a un DataFrame."""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj['Body'].read()))

# Cargar fact_ordenes
print("Cargando modelo dimensional desde silver...")
fact_ordenes = load_parquet_from_s3(
    BUCKET_NAME,
    f"{SILVER_PREFIX}/fact_ordenes_{FECHA_PROCESO}.parquet"
)
print(f"✓ fact_ordenes: {len(fact_ordenes):,} registros")

# Cargar dimensiones
dim_tiempo = load_parquet_from_s3(
    BUCKET_NAME,
    f"{SILVER_PREFIX}/dim_tiempo_{FECHA_PROCESO}.parquet"
)
print(f"✓ dim_tiempo: {len(dim_tiempo):,} registros")

dim_tipo_mant = load_parquet_from_s3(
    BUCKET_NAME,
    f"{SILVER_PREFIX}/dim_tipo_mantenimiento_{FECHA_PROCESO}.parquet"
)
print(f"✓ dim_tipo_mantenimiento: {len(dim_tipo_mant):,} registros")

dim_activos = load_parquet_from_s3(
    BUCKET_NAME,
    f"{SILVER_PREFIX}/dim_activos_{FECHA_PROCESO}.parquet"
)
print(f"✓ dim_activos: {len(dim_activos):,} registros")

## 5. Enriquecer Fact con Dimensiones

In [ ]:
# Join fact_ordenes con dimensiones para análisis
ordenes_enriquecidas = fact_ordenes.copy()

# Join con dim_tiempo (para fecha_inicio)
ordenes_enriquecidas = ordenes_enriquecidas.merge(
    dim_tiempo[['fecha_id', 'fecha', 'año', 'mes']],
    left_on='fecha_inicio_id',
    right_on='fecha_id',
    how='left'
).rename(columns={'fecha': 'fecha_inicio'})

# Join con dim_tipo_mantenimiento
ordenes_enriquecidas = ordenes_enriquecidas.merge(
    dim_tipo_mant[['tipo_mant_id', 'tipo_mantenimiento', 'categoria']],
    on='tipo_mant_id',
    how='left'
)

# Join con dim_activos
ordenes_enriquecidas = ordenes_enriquecidas.merge(
    dim_activos[['activo_id', 'nombre_activo']],
    on='activo_id',
    how='left'
)

print(f"=== Fact Enriquecida ===")
print(f"Total registros: {len(ordenes_enriquecidas):,}")
print(f"Columnas: {ordenes_enriquecidas.columns.tolist()}")
print(f"\nPrimeras 5 filas:")
print(ordenes_enriquecidas.head())

## 6. Calcular Métricas por Activo

In [ ]:
# Calcular métricas agregadas por activo
print("=== Calculando métricas por activo ===")

metricas_activo = ordenes_enriquecidas.groupby('activo_id').agg(
    nombre_activo=('nombre_activo', 'first'),
    num_ordenes=('orden_id', 'count'),
    total_horas_mantenimiento=('horas_trabajo', 'sum'),
    media_horas_mantenimiento=('horas_trabajo', 'mean'),
    num_preventivos=('tipo_mantenimiento', lambda x: (x == 'PREVENTIVO').sum()),
    num_correctivos=('tipo_mantenimiento', lambda x: (x == 'CORRECTIVO').sum()),
    num_predictivos=('tipo_mantenimiento', lambda x: (x == 'PREDICTIVO').sum()),
    num_completados=('estado', lambda x: (x == 'COMPLETADO').sum()),
    num_en_proceso=('estado', lambda x: (x == 'EN_PROCESO').sum())
).reset_index()

print(f"Métricas calculadas para {len(metricas_activo)} activos")
print(f"\nPrimeras 10 filas:")
print(metricas_activo.head(10))

## 7. Calcular Ratios y Porcentajes

In [ ]:
# Calcular ratios y porcentajes
metricas_activo['pct_preventivo'] = np.where(
    metricas_activo['num_ordenes'] > 0,
    (metricas_activo['num_preventivos'] / metricas_activo['num_ordenes']) * 100,
    0
)

metricas_activo['pct_correctivo'] = np.where(
    metricas_activo['num_ordenes'] > 0,
    (metricas_activo['num_correctivos'] / metricas_activo['num_ordenes']) * 100,
    0
)

metricas_activo['tasa_completitud'] = np.where(
    metricas_activo['num_ordenes'] > 0,
    (metricas_activo['num_completados'] / metricas_activo['num_ordenes']) * 100,
    0
)

print("=== Métricas con Ratios ===")
print(metricas_activo.head(10))

## 8. Calcular MTBM (Mean Time Between Maintenance)

In [ ]:
# Ordenar órdenes por activo y fecha de inicio para calcular MTBM
ordenes_sorted = ordenes_enriquecidas.sort_values(['activo_id', 'fecha_inicio']).reset_index(drop=True)

# Calcular tiempo entre mantenimientos consecutivos (en días)
ordenes_sorted['fecha_inicio_prev'] = ordenes_sorted.groupby('activo_id')['fecha_inicio'].shift(1)
ordenes_sorted['dias_desde_ultimo_mant'] = (
    (ordenes_sorted['fecha_inicio'] - ordenes_sorted['fecha_inicio_prev'])
    .dt.total_seconds() / (24 * 3600)
)

print("=== Tiempo entre Mantenimientos ===")
print(f"Media de días entre mantenimientos: {ordenes_sorted['dias_desde_ultimo_mant'].mean():.2f}")
print(f"Mediana: {ordenes_sorted['dias_desde_ultimo_mant'].median():.2f}")

# Calcular MTBM por activo
mtbm_por_activo = ordenes_sorted.groupby('activo_id').agg(
    mtbm_dias=('dias_desde_ultimo_mant', 'mean'),
    num_intervalos=('dias_desde_ultimo_mant', 'count')
).reset_index()

print(f"\n=== MTBM por Activo (primeros 10) ===")
print(mtbm_por_activo.head(10))

## 9. Consolidar KPIs Finales

In [ ]:
# Combinar métricas por activo con MTBM
kpis_finales = metricas_activo.merge(mtbm_por_activo, on='activo_id', how='left')

# Añadir información de periodo
kpis_finales['año_mes'] = MES_PROCESO
kpis_finales['_generado_timestamp'] = datetime.now().isoformat()

# Reordenar columnas
columnas_orden = [
    'año_mes',
    'activo_id',
    'nombre_activo',
    'num_ordenes',
    'total_horas_mantenimiento',
    'media_horas_mantenimiento',
    'num_preventivos',
    'num_correctivos',
    'num_predictivos',
    'pct_preventivo',
    'pct_correctivo',
    'num_completados',
    'num_en_proceso',
    'tasa_completitud',
    'mtbm_dias',
    'num_intervalos',
    '_generado_timestamp'
]

kpis_finales = kpis_finales[columnas_orden]

print(f"=== KPIs Finales Consolidados ===")
print(f"Total activos: {len(kpis_finales)}")
print(f"\nPrimeras 10 filas:")
print(kpis_finales.head(10))

## 10. Resumen Estadístico de KPIs

In [ ]:
print("=" * 60)
print("RESUMEN DE KPIs")
print("=" * 60)
print(f"\n📊 Total activos con mantenimiento: {len(kpis_finales)}")
print(f"\n🔧 Órdenes de Mantenimiento:")
print(f"  - Media de órdenes por activo: {kpis_finales['num_ordenes'].mean():.2f}")
print(f"  - Total órdenes procesadas: {kpis_finales['num_ordenes'].sum():,}")

print(f"\n⏱️  Horas de Mantenimiento:")
print(f"  - Media por activo: {kpis_finales['total_horas_mantenimiento'].mean():.2f} horas")
print(f"  - Total horas: {kpis_finales['total_horas_mantenimiento'].sum():.2f} horas")

print(f"\n📈 MTBM (Mean Time Between Maintenance):")
print(f"  - Promedio: {kpis_finales['mtbm_dias'].mean():.2f} días")
print(f"  - Mediana: {kpis_finales['mtbm_dias'].median():.2f} días")

print(f"\n🔄 Distribución de Mantenimientos:")
print(f"  - % Preventivo promedio: {kpis_finales['pct_preventivo'].mean():.2f}%")
print(f"  - % Correctivo promedio: {kpis_finales['pct_correctivo'].mean():.2f}%")
print(f"  - Tasa de completitud promedio: {kpis_finales['tasa_completitud'].mean():.2f}%")

print(f"\n🏆 Top 10 Activos con Más Horas de Mantenimiento:")
top_activos = kpis_finales.nlargest(10, 'total_horas_mantenimiento')[[
    'activo_id', 'nombre_activo', 'total_horas_mantenimiento', 'num_ordenes'
]]
print(top_activos.to_string(index=False))
print("=" * 60)

## 11. Calcular KPIs por Tipo de Mantenimiento

In [ ]:
# KPIs agregados por tipo de mantenimiento
kpis_por_tipo = ordenes_enriquecidas.groupby('tipo_mantenimiento').agg(
    num_ordenes=('orden_id', 'count'),
    total_horas=('horas_trabajo', 'sum'),
    horas_promedio=('horas_trabajo', 'mean'),
    duracion_promedio_horas=('duracion_horas', 'mean'),
    pct_completados=('estado', lambda x: (x == 'COMPLETADO').sum() / len(x) * 100 if len(x) > 0 else 0)
).reset_index()

# Añadir información de periodo
kpis_por_tipo['año_mes'] = MES_PROCESO
kpis_por_tipo['_generado_timestamp'] = datetime.now().isoformat()

print("=== KPIs por Tipo de Mantenimiento ===")
print(kpis_por_tipo)

## 12. Guardar KPIs en Gold/

In [ ]:
def save_parquet_to_s3(df, bucket, key, descripcion):
    """Guarda un DataFrame en S3 como Parquet."""
    temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
    temp_file.close()
    
    df.to_parquet(temp_file.name, index=False)
    s3_client.upload_file(temp_file.name, bucket, key)
    
    Path(temp_file.name).unlink()
    print(f"✓ {descripcion} → s3://{bucket}/{key} ({len(df):,} registros)")

print("=== Guardando KPIs en Gold ===")

# Guardar KPIs mensuales por activo
save_parquet_to_s3(
    kpis_finales,
    BUCKET_NAME,
    f"{GOLD_PREFIX}/kpis_mensuales_activos_{MES_PROCESO}.parquet",
    "KPIs mensuales por activo"
)

# Guardar KPIs por tipo de mantenimiento
save_parquet_to_s3(
    kpis_por_tipo,
    BUCKET_NAME,
    f"{GOLD_PREFIX}/kpis_por_tipo_{MES_PROCESO}.parquet",
    "KPIs por tipo de mantenimiento"
)

## 13. Resumen Final

In [ ]:
print("=" * 60)
print("RESUMEN - Silver → Gold")
print("=" * 60)
print(f"\n📥 ENTRADA")
print(f"  - Fuente: silver/obra-lineal/")
print(f"  - fact_ordenes: {len(fact_ordenes):,} registros")
print(f"  - dim_tiempo: {len(dim_tiempo):,} registros")
print(f"  - dim_tipo_mantenimiento: {len(dim_tipo_mant):,} registros")
print(f"  - dim_activos: {len(dim_activos):,} registros")

print(f"\n📊 KPIs GENERADOS")
print(f"  - KPIs mensuales por activo: {len(kpis_finales):,} activos")
print(f"  - KPIs por tipo de mantenimiento: {len(kpis_por_tipo):,} tipos")

print(f"\n💡 INSIGHTS CLAVE")
print(f"  - Activos monitoreados: {len(kpis_finales)}")
print(f"  - Total órdenes procesadas: {kpis_finales['num_ordenes'].sum():,}")
print(f"  - MTBM promedio: {kpis_finales['mtbm_dias'].mean():.1f} días")
print(f"  - % Preventivo vs Correctivo: {kpis_finales['pct_preventivo'].mean():.1f}% vs {kpis_finales['pct_correctivo'].mean():.1f}%")

print(f"\n✓ KPIs guardados en: gold/obra-lineal/")
print("=" * 60)
print("\n✅ Pipeline completado: Landing → Bronze → Silver → Gold")